In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
import json
from birddog.database import Database
from birddog.runtime import Runtime
from birddog.wiki import WIKI_NAMESPACE, _expand_link_target
from birddog.core import Page
from birddog.utility import fetch_url

2026-01-02 13:54:04,701 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.
2026-01-02 13:54:04,869 [INFO] Translation is enabled. Using GCP translator
2026-01-02 13:54:04,870 [INFO] Using Google Cloud translation API
2026-01-02 13:54:04,870 [INFO] GoogleCloudTranslator using REST API


In [3]:
runtime = Runtime()

2026-01-02 13:54:05,255 [INFO] PageUpdateManager.init(): detect_environment==local
2026-01-02 13:54:05,395 [INFO] KillSwitch: loading thresholds from resources/kill_thresholds.json
2026-01-02 13:54:05,396 [INFO] Runtime: truncating log history before 2025-11-03 20:54:05.396233+00:00


In [4]:
db = Database()

In [5]:
def _format_date(date):
    d = date.split(",")
    return f"{d[0]}-{d[1]}-{d[2]} {d[3]}:00+00:00"

In [6]:
def _page_title(page):
    return page.page["title"]["uk"]

In [7]:
def form_page_record(page):
    page_data = page.page
    
    result = { "title": _page_title(page) }
    result["description_uk"] = page_data["description"]["uk"]
    desc = page_data["description"].get("en")
    if desc:
        result["description"] = desc
    result["availability"] = "linked"
    result["level"] = page.kind
    result["reference_date"] = _format_date(page_data["lastmod"])
    result["label"] = page.display_name if page.kind == "archive" else page.id
    dates = page_data.get("dates")
    if dates:
        result["years"] = dates["uk"]
    result["source_type"] = "wiki"
    
    return result

In [8]:
def _availability(cell):
    return "linked" if cell.get("exists", False) else "redlinked"
    
def child_titles(page):
    result = []
    prefix = f"/wiki/{WIKI_NAMESPACE}:"
    for child in page.children:
        if child:
            for cell in child:
                link = cell.get("link")
                if link and link.startswith(prefix):
                    result.append({
                        "title": link.replace(prefix, ""),
                        "availability": _availability(cell),
                    })
                    break
    return result

In [9]:
def detect_changes(db, table_name, records, key="title"):
    if isinstance(records, dict):
        records = [ records ]
        singleton = True
    else:
        singleton = False
    id_map = db.lookup(table_name, {record[key] for record in records})
    current_records = db.read(table_name, list(id_map.values()))
    current_record_dict = {
        rec["Id"]: rec 
        for rec in db.read(table_name, list(id_map.values()))
        }
    update = []
    for record in records:
        rec_id = id_map.get(record[key])
        if rec_id:
            current_rec = current_record_dict[rec_id]
            for k, v in record.items():
                if not k in current_rec or current_rec[k] != v:
                    #if k in current_rec:
                    #    print(f"data mismatch: {current_rec[k]} != {v}")
                    update.append(record)
                    break
        else:
            update.append(record)
    for record in update:
        record["birddog_alert"] = True
    if singleton:
        return update[0] if update else None
    return update

In [10]:
def link_children(db, page):
    parent_title = _page_title(page)
    child_title_records = child_titles(page)
    # ensure child records exist
    child_ids = db.write("Pages", child_title_records)
    parent_id = db.lookup("Pages", parent_title)
    if not parent_id:
        raise ValueError(f"cannot find record for parent (title={parent_title})")
    print(f"attaching {len(child_ids)} child links")
    db.create_links(
        "Pages", "children", 
        parent_id, child_ids)

In [11]:
def update_pages(db, pages):
    if not isinstance(pages, (list, tuple)):
        if not isinstance(pages, Page):
            raise ValueError("must be list/tuple of Page objects or singleton Page object")
        singleton = True
        pages = [ pages ]
    else:
        singleton = False
    update = detect_changes(db, "Pages", form_page_record(page))
    if update:
        result = db.write("Pages", update)
        if singleton:
            return result[0]
        return result
    else:
        print("nothing to update")
    return None

In [12]:
page = runtime.lookup_by_address("DAZHO","D","3","1", "6")
update_pages(db, page)
page.page

2026-01-02 13:54:12,714 [INFO] PageLRU.lookup_by_address(DAZHO, D, 3, 1, 6), archive=Архів:ДАЖО/Д
2026-01-02 13:54:12,715 [INFO] PageLRU.lookup(Архів:ДАЖО/Д): miss
2026-01-02 13:54:12,980 [INFO] fetch_url: 1 requests in last 60s → 0.02 req/s
2026-01-02 13:54:12,993 [INFO] PageLRU.lookup(Архів:ДАЖО/3): miss
2026-01-02 13:54:13,228 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1): miss
2026-01-02 13:54:13,460 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/6): miss
nothing to update


{'title': {'uk': 'ДАЖО/3/1/6', 'en': 'EVEN/3/1/6'},
 'template': {'uk': 'Архіви/справа', 'en': 'Archives/case'},
 'revid': 884030,
 'description': {'uk': "Прохання міщанина І.А. Гехтмана про оцінку його кам'яного будинку в м. Житомирі",
  'en': 'Request from a citizen I.A. Gekhtman for an assessment of his stone house in the city of Zhytomyr'},
 'dates': {'uk': '1817', 'en': '1817'},
 'notes': {'commons_links': ["https://commons.wikimedia.org/wiki/File:ДАЖО_3-1-6_Прохання_міщанина_І.А._Гехтмана_про_оцінку_його_кам'яного_будинку_в_м._Житомирі_(1817).pdf"]},
 'other_links': {'commons_links': [],
  'category_links': ['Категорія:Житомир'],
  'internal_links': ["File:ДАЖО 3-1-6 Прохання міщанина І.А. Гехтмана про оцінку його кам'яного будинку в м. Житомирі (1817).pdf"],
  'external_links': []},
 'tables': [],
 'link': 'https://uk.wikisource.org/wiki/Архів:ДАЖО/3/1/6',
 'doc_link': "https://uk.wikisource.org/wiki/File:ДАЖО_3-1-6_Прохання_міщанина_І.А._Гехтмана_про_оцінку_його_кам'яного_будин

In [13]:
def extract_page_links(page):
    result = set()
    page_data = page.page
    for key in ["notes", "other_links"]:
        for k, v in page_data.get(key, {}).items():
            if k == "category_links":
                continue
            for item in v:
                # remove trailing label
                item = item.split("|")[0]
                if not item.startswith("http"):
                    # seems to be a link target, expand it
                    item = _expand_link_target(item, page.title)
                result.add(item)
    doc_link = page_data.get("doc_link")
    if doc_link:
        result.add(doc_link)
    return list(result)

In [14]:
extract_page_links(page)

["https://uk.wikisource.org/wiki/File:ДАЖО_3-1-6_Прохання_міщанина_І.А._Гехтмана_про_оцінку_його_кам'яного_будинку_в_м._Житомирі_(1817).pdf",
 "https://commons.wikimedia.org/wiki/File:ДАЖО_3-1-6_Прохання_міщанина_І.А._Гехтмана_про_оцінку_його_кам'яного_будинку_в_м._Житомирі_(1817).pdf"]

In [15]:
page = runtime.lookup_by_address("DAZHO","D","3","1")
print(len(page.children))
update_pages(db, page)
extract_page_links(page)

2026-01-02 13:54:21,916 [INFO] PageLRU.lookup_by_address(DAZHO, D, 3, 1, None), archive=Архів:ДАЖО/Д
2026-01-02 13:54:21,917 [INFO] PageLRU.lookup(Архів:ДАЖО/Д): hit
2026-01-02 13:54:21,917 [INFO] PageLRU.lookup(Архів:ДАЖО/3): hit
2026-01-02 13:54:21,918 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1): hit
16
nothing to update


['https://commons.wikimedia.org/wiki/file:ДАЖО_фонд_3_опис_1.pdf']

In [16]:
child_pages = [runtime.lookup_by_title(entry["title"]) for entry in child_titles(page)]
update_pages(db, child_pages)
link_children(db, page)

2026-01-02 13:54:23,542 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/6): hit
2026-01-02 13:54:23,543 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/30): miss
2026-01-02 13:54:23,780 [INFO] fetch_url: 5 requests in last 60s → 0.08 req/s
2026-01-02 13:54:23,784 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/32): miss
2026-01-02 13:54:24,003 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/42): miss
2026-01-02 13:54:24,236 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/48): miss
2026-01-02 13:54:24,463 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/49): miss
2026-01-02 13:54:24,700 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/65): miss
2026-01-02 13:54:24,917 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/67): miss
2026-01-02 13:54:25,155 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/68): miss
2026-01-02 13:54:25,381 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/73): miss
2026-01-02 13:54:25,621 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/76): miss
2026-01-02 13:54:25,842 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/78): miss
2026-01-02 13:54:26,063 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1/79): miss
2

In [17]:
page = runtime.lookup_by_address("DAZHO","D","3")
print(len(page.children))
update_pages(db, page)
extract_page_links(page)

2026-01-02 13:54:31,083 [INFO] PageLRU.lookup_by_address(DAZHO, D, 3, None, None), archive=Архів:ДАЖО/Д
2026-01-02 13:54:31,084 [INFO] PageLRU.lookup(Архів:ДАЖО/Д): hit
2026-01-02 13:54:31,086 [INFO] PageLRU.lookup(Архів:ДАЖО/3): hit
1
nothing to update


[]

In [18]:
child_pages = [runtime.lookup_by_title(entry["title"]) for entry in child_titles(page)]
update_pages(db, child_pages)
link_children(db, page)

2026-01-02 13:54:33,181 [INFO] PageLRU.lookup(Архів:ДАЖО/3/1): hit
nothing to update
attaching 1 child links


In [19]:
page = runtime.lookup_by_address("DAZHO","D")
print(len(page.children))
update_pages(db, page)
extract_page_links(page)

2026-01-02 13:54:39,185 [INFO] PageLRU.lookup_by_address(DAZHO, D, None, None, None), archive=Архів:ДАЖО/Д
2026-01-02 13:54:39,186 [INFO] PageLRU.lookup(Архів:ДАЖО/Д): hit
475
nothing to update


['https://zhytomyr.archives.gov.ua/?page_id=276',
 'https://uk.wikisource.org/wiki/Архів:ДАЖО',
 'https://uk.wikisource.org/wiki/Архів:ДАЖО/Р']

In [20]:
from urllib.parse import urlparse, unquote

def parse_mediawiki_file_url(url):
    """
    Parse a MediaWiki file URL and return:
      {
        "title": "File:...",
        "source": "commons" | "wikisource" | "other",
        "link": canonical_link_or_original
      }

    If source == "other", title will be None and link is the original URL.
    """
    parsed = urlparse(url)
    host = parsed.netloc.lower()
    path = parsed.path

    # Must be a /wiki/ URL to extract a title
    if not path.startswith("/wiki/"):
        return {
            "title": None,
            "source": "other",
            "link": url,
        }

    # Extract and decode title
    title = unquote(path[len("/wiki/"):])

    # Wikimedia Commons
    if host == "commons.wikimedia.org":
        return {
            "title": title,
            "source": "commons",
            "link": f"https://commons.wikimedia.org/wiki/{title}",
        }

    # Wikisource (any language subdomain)
    if host.endswith(".wikisource.org"):
        return {
            "title": title,
            "source": "wikisource",
            "link": f"https://{host}/wiki/{title}",
        }

    # Anything else
    return {
        "title": None,
        "source": "other",
        "link": url,
    }


In [21]:
def fetch_mediawiki_file_metadata_batch(titles, source, thumbnail_width = 300):
    """
    Batch-fetch MediaWiki file metadata for one or more File: titles.

    Returns a dict keyed by the *caller-supplied* titles (trimmed), with metadata
    containing both:
      - "requested_title": as supplied by the caller (after trimming)
      - "api_title": the title returned by the API for the page (canonical/normalized)

    The MediaWiki API may normalize titles (underscores -> spaces, punctuation, etc.).
    This function reads query.normalized[] and uses it to preserve a stable mapping
    back to the caller's input.

    Parameters
    ----------
    titles : str | list[str]
        One title (e.g. "Файл:…pdf" or "File:…pdf") or a list of titles.
    source : str
        "commons" or "wikisource".
    thumbnail_width : int
        Requested thumbnail width in pixels.

    Returns
    -------
    dict[str, dict]
        Mapping of requested_title -> metadata dict.
    """
    # Resolve API endpoint
    if source == "commons":
        api = "https://commons.wikimedia.org/w/api.php"
    elif source == "wikisource":
        api = "https://uk.wikisource.org/w/api.php"
    else:
        raise ValueError(f"Unsupported source: {source}")

    # Normalize input to list (preserve caller strings for keys)
    if isinstance(titles, str):
        requested_titles: List[str] = [titles.strip()]
    else:
        requested_titles = [t.strip() for t in titles if t and t.strip()]

    if not requested_titles:
        return {}

    params = {
        "action": "query",
        "format": "json",
        "prop": "imageinfo",
        "titles": "|".join(requested_titles),
        "iiprop": "timestamp|size|mime|mediatype|url|sha1",
        "iiurlwidth": thumbnail_width,
        "iilimit": 1,
    }
    data = fetch_url(api, params=params, json=True)
    query = data.get("query", {})

    # 1) Build mapping: input title -> normalized title (API "to")
    # If a title is not normalized by the API, map to itself.
    normalized_map = {t: t for t in requested_titles}
    for item in query.get("normalized", []):
        frm = item.get("from")
        to = item.get("to")
        if frm and to:
            normalized_map[frm] = to

    # 2) Build reverse index: api_title -> page object
    pages = query.get("pages", {})
    pages_by_title = {}
    for _pageid, page in pages.items():
        t = page.get("title")
        if t:
            pages_by_title[t] = page

    # 3) For each requested title, resolve to the API-normalized title, then to page
    results = {}
    for req_title in requested_titles:
        api_lookup_title = normalized_map.get(req_title, req_title)
        page = pages_by_title.get(api_lookup_title)

        # page may be missing even if present in pages dict
        if not page or page.get("missing") or not page.get("imageinfo"):
            results[req_title] = None
            continue

        ii = page["imageinfo"][0]
        record = {
            "title": page.get("title"),
            "link": ii.get("url"),
            "source": source,
            "timestamp": ii.get("timestamp"),
            "byte_size": ii.get("size"),
            "mimetype": ii.get("mime"),
            "mediatype": ii.get("mediatype"),
            "width": ii.get("width"),
            "height": ii.get("height"),
            "page_count": ii.get("pagecount"),
            "sha1_hash": ii.get("sha1"),
            "description_url": ii.get("descriptionurl"),
        }
        if "thumburl" in ii:
            record["thumb_url"] = ii.get("thumburl")
            record["thumb_width"] = ii.get("thumbwidth")
            record["thumb_height"] = ii.get("thumbheight")
        results[req_title] = record

    return results


In [22]:
files = [
    "https://commons.wikimedia.org/wiki/File:%D0%94%D0%90%D0%A5%D0%B5%D0%9E_%D0%A4%D0%BE%D0%BD%D0%B4_1_%D0%9E%D0%BF%D0%B8%D1%81%D0%B8_1,_2.pdf",
    "https://uk.wikisource.org/wiki/%D0%A4%D0%B0%D0%B9%D0%BB:%D0%94%D0%90%D0%A5%D0%B5%D0%9E_1-1-11._1842._%D0%A0%D0%B0%D0%BF%D0%BE%D1%80%D1%82%D1%8B_%D0%BF%D0%BE%D0%BB%D0%B8%D1%86%D0%BC%D0%B5%D0%B9%D1%81%D1%82%D0%B5%D1%80%D0%BE%D0%B2_%D0%B8_%D0%BF%D0%B5%D1%80%D0%B5%D0%BF%D0%B8%D1%81%D0%BA%D0%B0_%D1%81_%D0%9A%D0%B8%D0%B5%D0%B2%D1%81%D0%BA%D0%B8%D0%BC,_%D0%9F%D0%BE%D0%B4%D0%BE%D0%BB%D1%8C%D1%81%D0%BA%D0%B8%D0%BC_%D0%B8_%D0%B4%D1%80%D1%83%D0%B3%D0%B8%D0%BC%D0%B8_%D0%B3%D1%83%D0%B1%D0%B5%D1%80%D0%BD%D1%81%D0%BA%D0%B8%D0%BC%D0%B8_%D0%BF%D1%80%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%B8%D1%8F%D0%BC%D0%B8.pdf",
    "https://uk.wikisource.org/wiki/%D0%A4%D0%B0%D0%B9%D0%BB:%D0%94%D0%90%D0%A5%D0%B5%D0%9E_6-2-1._1900-1901._%D0%92%D1%96%D0%B4%D0%BE%D0%BC%D0%BE%D1%81%D1%82%D1%96_%D0%BF%D1%80%D0%BE_%D1%81%D1%82%D0%B0%D0%BD_%D1%88%D0%BA%D1%96%D0%BB_%D0%B2_%D0%A5%D0%B5%D1%80%D1%81%D0%BE%D0%BD%D1%81%D1%8C%D0%BA%D0%BE%D0%BC%D1%83_%D0%BF%D0%BE%D0%B2%D1%96%D1%82%D1%96_%D1%82%D0%B0_%D1%82%D0%B0%D0%B1%D0%BB%D0%B8%D1%86%D1%96_%D1%80%D0%BE%D0%B7%D0%BC%D1%96%D1%80%D1%96%D0%B2_%D0%BF%D0%BE%D1%81%D1%96%D0%B2%D0%BD%D0%BE%D1%97_%D0%BF%D0%BB%D0%BE%D1%89%D1%96_%D0%B7_%D1%83%D1%80%D0%B0%D1%85%D1%96%D0%B2%D0%B0%D0%BD%D0%BD%D1%8F%D0%BC_%D0%BA%D1%96%D0%BB%D1%8C%D0%BA%D0%BE%D1%81%D1%82%D1%96_%D1%81%D1%96%D0%BC%D0%B5%D0%B9.pdf", 
]



In [23]:
p=[parse_mediawiki_file_url(f) for f in extract_page_links(page)]
p

[{'title': None,
  'source': 'other',
  'link': 'https://zhytomyr.archives.gov.ua/?page_id=276'},
 {'title': 'Архів:ДАЖО',
  'source': 'wikisource',
  'link': 'https://uk.wikisource.org/wiki/Архів:ДАЖО'},
 {'title': 'Архів:ДАЖО/Р',
  'source': 'wikisource',
  'link': 'https://uk.wikisource.org/wiki/Архів:ДАЖО/Р'}]

In [24]:
fetch_mediawiki_file_metadata_batch([v["title"] for v in p if v["source"]=="wikisource"], source="wikisource")

2026-01-02 13:55:00,618 [INFO] fetch_url: 20 requests in last 60s → 0.33 req/s


{'Архів:ДАЖО': None, 'Архів:ДАЖО/Р': None}

In [25]:
parse1 = parse_mediawiki_file_url(files[1])
parse2 = parse_mediawiki_file_url(files[2])
result=fetch_mediawiki_file_metadata_batch([parse1["title"],parse2["title"],"foo"], source=parse1["source"])

2026-01-02 13:55:11,316 [INFO] fetch_url: 21 requests in last 60s → 0.35 req/s


In [26]:
import json
for k,v in result.items():
    print(f"---- {k} ----")
    print(json.dumps(v, indent=4))

---- Файл:ДАХеО_1-1-11._1842._Рапорты_полицмейстеров_и_переписка_с_Киевским,_Подольским_и_другими_губернскими_правлениями.pdf ----
{
    "title": "\u0424\u0430\u0439\u043b:\u0414\u0410\u0425\u0435\u041e 1-1-11. 1842. \u0420\u0430\u043f\u043e\u0440\u0442\u044b \u043f\u043e\u043b\u0438\u0446\u043c\u0435\u0439\u0441\u0442\u0435\u0440\u043e\u0432 \u0438 \u043f\u0435\u0440\u0435\u043f\u0438\u0441\u043a\u0430 \u0441 \u041a\u0438\u0435\u0432\u0441\u043a\u0438\u043c, \u041f\u043e\u0434\u043e\u043b\u044c\u0441\u043a\u0438\u043c \u0438 \u0434\u0440\u0443\u0433\u0438\u043c\u0438 \u0433\u0443\u0431\u0435\u0440\u043d\u0441\u043a\u0438\u043c\u0438 \u043f\u0440\u0430\u0432\u043b\u0435\u043d\u0438\u044f\u043c\u0438.pdf",
    "link": "https://upload.wikimedia.org/wikipedia/commons/7/75/%D0%94%D0%90%D0%A5%D0%B5%D0%9E_1-1-11._1842._%D0%A0%D0%B0%D0%BF%D0%BE%D1%80%D1%82%D1%8B_%D0%BF%D0%BE%D0%BB%D0%B8%D1%86%D0%BC%D0%B5%D0%B9%D1%81%D1%82%D0%B5%D1%80%D0%BE%D0%B2_%D0%B8_%D0%BF%D0%B5%D1%80%D0%B5%D0%BF%D0%B8%D1%

In [27]:
parse = parse_mediawiki_file_url(files[0])
fetch_mediawiki_file_metadata_batch(parse["title"], source=parse["source"])

{'File:ДАХеО_Фонд_1_Описи_1,_2.pdf': {'title': 'File:ДАХеО Фонд 1 Описи 1, 2.pdf',
  'link': 'https://upload.wikimedia.org/wikipedia/commons/3/3f/%D0%94%D0%90%D0%A5%D0%B5%D0%9E_%D0%A4%D0%BE%D0%BD%D0%B4_1_%D0%9E%D0%BF%D0%B8%D1%81%D0%B8_1%2C_2.pdf',
  'source': 'commons',
  'timestamp': '2022-02-20T00:00:18Z',
  'byte_size': 25693630,
  'mimetype': 'application/pdf',
  'mediatype': 'OFFICE',
  'width': 1237,
  'height': 1750,
  'page_count': 30,
  'sha1_hash': '7242f40218ea51b2d0be3fa29d9750c3a1b61087',
  'description_url': 'https://commons.wikimedia.org/wiki/File:%D0%94%D0%90%D0%A5%D0%B5%D0%9E_%D0%A4%D0%BE%D0%BD%D0%B4_1_%D0%9E%D0%BF%D0%B8%D1%81%D0%B8_1,_2.pdf',
  'thumb_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/%D0%94%D0%90%D0%A5%D0%B5%D0%9E_%D0%A4%D0%BE%D0%BD%D0%B4_1_%D0%9E%D0%BF%D0%B8%D1%81%D0%B8_1%2C_2.pdf/page1-300px-%D0%94%D0%90%D0%A5%D0%B5%D0%9E_%D0%A4%D0%BE%D0%BD%D0%B4_1_%D0%9E%D0%BF%D0%B8%D1%81%D0%B8_1%2C_2.pdf.jpg',
  'thumb_width': 300,
  'thumb_height':

In [28]:
parse["title"]

'File:ДАХеО_Фонд_1_Описи_1,_2.pdf'

In [29]:
parse2


{'title': 'Файл:ДАХеО_6-2-1._1900-1901._Відомості_про_стан_шкіл_в_Херсонському_повіті_та_таблиці_розмірів_посівної_площі_з_урахіванням_кількості_сімей.pdf',
 'source': 'wikisource',
 'link': 'https://uk.wikisource.org/wiki/Файл:ДАХеО_6-2-1._1900-1901._Відомості_про_стан_шкіл_в_Херсонському_повіті_та_таблиці_розмірів_посівної_площі_з_урахіванням_кількості_сімей.pdf'}

In [30]:
page = runtime.lookup_by_address("DAKO", "D", "384", "9")

2026-01-02 13:55:29,122 [INFO] PageLRU.lookup_by_address(DAKO, D, 384, 9, None), archive=Архів:ДАКО/Д
2026-01-02 13:55:29,123 [INFO] PageLRU.lookup(Архів:ДАКО/Д): miss
2026-01-02 13:55:29,344 [INFO] fetch_url: 4 requests in last 60s → 0.07 req/s
2026-01-02 13:55:29,358 [INFO] PageLRU.lookup(Архів:ДАКО/384): miss
2026-01-02 13:55:29,582 [INFO] PageLRU.lookup(Архів:ДАКО/384/9): miss


In [31]:
_blocklist = [
    "FSMosaicTreeLogo",
    "familysearch.org",
]

def _allowed_link(link):
    return all([item not in link for item in _blocklist])

def check_title_links(title):
    page = runtime.lookup_by_title(title)
    l=extract_page_links(page)
    return [x for x in l if _allowed_link(x)]

In [32]:
with open("var/title_sample.json") as file:
    title_list = json.loads(file.read())
title_list = list(set(title_list))

In [33]:
len(title_list)

645

In [ ]:
result = []
for title in title_list:
    links = check_title_links(title)
    result.extend(links)
result = sorted(list(set(result)))
print(len(result))

2026-01-02 13:56:00,713 [INFO] PageLRU.lookup(Архів:ЦДІАК/1166): miss
2026-01-02 13:56:00,974 [INFO] fetch_url: 6 requests in last 60s → 0.10 req/s
2026-01-02 13:56:00,980 [INFO] PageLRU.lookup(Архів:ДАВоО/35): miss
2026-01-02 13:56:01,203 [INFO] PageLRU.lookup(Архів:ДАВіО/256): miss
2026-01-02 13:56:01,431 [INFO] PageLRU.lookup(Архів:ІР_НБУВ/223): miss
2026-01-02 13:56:01,665 [INFO] PageLRU.lookup(Архів:ДАПО/1072/1/1): miss
2026-01-02 13:56:01,904 [INFO] PageLRU.lookup(Архів:ДАКО/3): miss
2026-01-02 13:56:02,126 [INFO] PageLRU.lookup(Архів:ДАВіО/815): miss
2026-01-02 13:56:02,353 [INFO] PageLRU.lookup(Архів:ДАІФО/Д): miss
2026-01-02 13:56:02,589 [INFO] PageLRU.lookup(Архів:ДАХмО/592/1): miss
2026-01-02 13:56:02,810 [INFO] PageLRU.lookup(Архів:ЦДАВО/571/1): miss
2026-01-02 13:56:03,040 [INFO] PageLRU.lookup(Архів:ДАЖО/118): miss
2026-01-02 13:56:03,258 [INFO] PageLRU.lookup(Архів:ЦДІАК/131): miss
2026-01-02 13:56:03,482 [INFO] PageLRU.lookup(Архів:ДАДнО/Р-6508/1/15): miss
2026-01-02 13

In [ ]:
def extract_hosts(links):
    result = set()
    for link in links:
        parsed = urlparse(link)
        host = parsed.netloc.lower()
        result.add(host)
    return sorted(list(result))

In [ ]:
extract_hosts(result)

In [ ]:
def extract_suffixes(links):
    result = set()
    for link in links:
        parsed = urlparse(link)
        path = parsed.path
        if "/" in path:
            tail = path.rsplit("/", 1)[-1]
            if "." in tail:
                suffix = tail.rsplit(".", 1)[-1]
                result.add(suffix)
    return sorted(list(result))

In [ ]:
extract_suffixes(result)